# Taller 1. Formulación y resolución de un modelo en red

**Investigación de Operaciones**
Doctorado en Ingeniería, consorcio UV–UTA
Jueves 10 de septiembre de 2026, Bloque B

---

**Integrantes:**
**Instancia asignada:**
**Fecha:**

---

Esta plantilla entrega la estructura común a las cuatro instancias. Lo que falta —y es lo que se
evalúa— son la formulación, la interpretación y la verificación. Los bloques marcados con
`# TODO` deben completarse.

Regla dura del taller: **ningún dato numérico se escribe dentro del modelo**. Todo se lee de los
archivos CSV de la carpeta `datos/`.


## Paso 0. Verificación del entorno


In [1]:
import sys
print("Python", sys.version.split()[0])

import pyomo.environ as pyo
print("Pyomo", pyo.__version__ if hasattr(pyo, "__version__") else "instalado")

solver = pyo.SolverFactory("appsi_highs")
print("HiGHS disponible:", solver.available())

import pandas as pd
print("pandas", pd.__version__)


Python 3.14.7
Pyomo instalado
HiGHS disponible: True
pandas 3.0.5


## Paso 1. Lectura de los datos

Ajuste `CARPETA` a la instancia que le fue asignada. El separador de los CSV es el punto y coma.


In [2]:
from pathlib import Path
import pandas as pd

CARPETA = Path("datos")          # TODO: apunte a la carpeta de su instancia

# Instancias A y B: nodos.csv y arcos.csv
# Instancia C: demanda.csv y parametros.csv  -> hay que construir la red expandida en el tiempo
# Instancia D: tiempos.csv                   -> hay que construir la red bipartita

nodos = pd.read_csv(CARPETA / "nodos.csv", sep=";")
arcos = pd.read_csv(CARPETA / "arcos.csv", sep=";")
display(nodos)
display(arcos)


,nodo,descripcion,flujo_exogeno_t_mes,tipo
0,F1,Faena Norte (planta concentradora),12000,oferta
1,F2,Faena Centro,9000,oferta
2,F3,Faena Sur,7000,oferta
3,A1,Acopio Calama,0,transbordo
4,A2,Acopio Copiapó,0,transbordo
5,P1,Puerto Angamos,-11000,demanda
6,P2,Puerto Chañaral,-8000,demanda
7,P3,Puerto Ventanas,-6000,demanda


,origen,destino,costo_peso_por_t,cap_min_t,cap_max_t
0,F1,A1,4200,0,10000
1,F1,A2,7800,0,6000
2,F2,A1,5100,0,8000
3,F2,A2,4600,0,8000
4,F3,A1,8400,0,4000
5,F3,A2,3900,0,7000
6,A1,P1,2800,0,12000
7,A1,P2,6100,0,6000
8,A1,P3,9200,0,5000
9,A2,P2,3300,0,9000


## Paso 2. Estructura de la red

Antes de programar nada, escriba en palabras qué representa cada nodo y cada arco, y con qué
unidades. Un modelo cuyas unidades no cierran está mal aunque el solver entregue un número.


In [ ]:
# TODO: complete los tres diccionarios

N = {
    'F1': "Faena Norte (planta concentradora)",
    'F2': "Faena Centro",
    'F3': "Faena Sur",
    'A1': "Acopio Calama",
    'A2': "Acopio Copiapó",
    'P1': "Puerto Angamos",
    'P2': "Puerto Chañaral",
    'P3': "Puerto Ventanas"
}      # conjunto de nodos
q = {
    'F1': 12000,    # Oferta Faena Norte
    'F2': 9000,     # Oferta Faena Centro
    'F3': 7000,     # Oferta Faena Sur
    'A1': 0,        # Transbordo Acopio Calama
    'A2': 0,        # Transbordo Acopio Copiapó
    'P1': -11000,   # Demanda Puerto Angamos
    'P2': -8000,    # Demanda Puerto Chañaral
    'P3': -6000     # Demanda Puerto Ventanas
}      # flujo exógeno por nodo: positivo oferta, negativo demanda, cero transbordo
A = {
    # Arcos desde Faenas a Acopios
    ('F1', 'A1'): (4200, 0, 10000),
    ('F1', 'A2'): (7800, 0, 6000),
    ('F2', 'A1'): (5100, 0, 8000),
    ('F2', 'A2'): (4600, 0, 8000),
    ('F3', 'A1'): (8400, 0, 4000),
    ('F3', 'A2'): (3900, 0, 7000),
    
    # Arcos desde Acopios a Puertos
    ('A1', 'P1'): (2800, 0, 12000),
    ('A1', 'P2'): (6100, 0, 6000),
    ('A1', 'P3'): (9200, 0, 5000),
    ('A2', 'P2'): (3300, 0, 9000),
    ('A2', 'P3'): (5700, 0, 7000),
    
    # Rutas directas (Faena -> Puerto)
    ('F1', 'P1'): (8900, 0, 4000),
    ('F3', 'P3'): (9600, 0, 3000)
}      # arcos: (i, j) -> (costo, cota_inferior, cota_superior)

# Comprobación imprescindible antes de seguir:
# si todas las restricciones son igualdades, la suma de los q_i debe ser cero.
print("suma de los flujos exógenos:", sum(q.values()))


## Paso 3. El modelo

$$\min \; \sum_{(i,j)\in A} c_{ij}\,x_{ij}
\quad\text{s.a.}\quad
\sum_{j} x_{ij} - \sum_{k} x_{ki} = q_i \;\; \forall i \in N,
\qquad l_{ij} \le x_{ij} \le u_{ij}$$

Note que las variables se declaran **continuas**. No se impone integralidad: si el modelo es
realmente de red y el lado derecho es entero, la solución saldrá entera por sí sola. Comprobarlo es
parte del taller.


In [ ]:
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.N = pyo.Set(initialize=list(N))
m.A = pyo.Set(initialize=list(A), dimen=2)

m.x = pyo.Var(m.A, domain=pyo.NonNegativeReals,
              bounds=lambda m, i, j: (A[(i, j)][1], A[(i, j)][2]))

m.obj = pyo.Objective(expr=sum(A[a][0] * m.x[a] for a in m.A), sense=pyo.minimize)

def balance(m, n):
    sale  = sum(m.x[i, j] for (i, j) in m.A if i == n)
    entra = sum(m.x[i, j] for (i, j) in m.A if j == n)
    # TODO: decida para qué nodos corresponde igualdad y para cuáles desigualdad,
    #       y justifique esa decisión en el informe.
    return sale - entra == q[n]

m.bal = pyo.Constraint(m.N, rule=balance)

# Los valores duales deben declararse ANTES de resolver, o no se importan.
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

print("variables:", len(m.A), " restricciones:", len(m.N))


## Paso 4. Resolución


In [ ]:
res = pyo.SolverFactory("appsi_highs").solve(m)
print("condición de término:", res.solver.termination_condition)
print("valor óptimo: ", pyo.value(m.obj))

flujos = {a: pyo.value(m.x[a]) for a in m.A}
for a, v in sorted(flujos.items()):
    if v > 1e-6:
        print(f"   {a[0]:>10} -> {a[1]:<10} {v:>12,.2f}   de {A[a][2]:,.0f} de capacidad")


## Paso 5. Verificación de la conservación de flujo

Esta comprobación es obligatoria y vale puntaje. No basta con afirmar que el solver la respetó.


In [ ]:
print(f"{'nodo':<10}{'sale':>12}{'entra':>12}{'neto':>12}{'exigido':>12}   estado")
todo_ok = True
for n in N:
    sale  = sum(v for (i, j), v in flujos.items() if i == n)
    entra = sum(v for (i, j), v in flujos.items() if j == n)
    neto  = sale - entra
    ok = abs(neto - q[n]) < 1e-6 or (q[n] > 0 and neto <= q[n] + 1e-6)
    todo_ok &= ok
    print(f"{n:<10}{sale:>12,.2f}{entra:>12,.2f}{neto:>12,.2f}{q[n]:>12,.2f}   {'ok' if ok else 'ERROR'}")
print("\nconservación de flujo verificada en todos los nodos:", todo_ok)


## Paso 6. Integralidad

¿Salieron enteros los flujos? ¿Por qué? La respuesta debe nombrar la propiedad de la matriz y la
condición sobre el lado derecho, no limitarse a constatar el hecho.


In [ ]:
no_enteros = {a: v for a, v in flujos.items() if abs(v - round(v)) > 1e-6}
print("flujos no enteros:", len(no_enteros))
if no_enteros:
    print(no_enteros)

# TODO: construya la matriz de incidencia nodo-arco y calcule el determinante de al menos
#       tres submatrices cuadradas. Verifique que todos caen en {0, 1, -1}.


## Paso 7. Valores duales

Informe cada dual con su **unidad** y declare la **convención de signos** que usa. Un dual sin
unidad no es interpretable y el informe pierde puntaje.


In [ ]:
for n in N:
    print(f"{n:<10} {m.dual[m.bal[n]]:>14,.2f}")

# TODO: interprete. ¿Qué significa el dual de un nodo de demanda? ¿Y el de uno con holgura?


## Paso 8. Análisis de sensibilidad

La forma más segura y más transparente de obtener el valor de una capacidad es **volver a
resolver** con esa capacidad modificada, y comparar. Es lo que hace la función siguiente.


In [ ]:
def resolver(A_mod, q_mod=None):
    """Resuelve una variante del modelo y devuelve (valor óptimo, flujos)."""
    q2 = q_mod if q_mod is not None else q
    mm = pyo.ConcreteModel()
    mm.A = pyo.Set(initialize=list(A_mod), dimen=2)
    mm.N = pyo.Set(initialize=list(q2))
    mm.x = pyo.Var(mm.A, domain=pyo.NonNegativeReals,
                   bounds=lambda mm, i, j: (A_mod[(i, j)][1], A_mod[(i, j)][2]))
    mm.obj = pyo.Objective(expr=sum(A_mod[a][0] * mm.x[a] for a in mm.A), sense=pyo.minimize)
    def bal(mm, n):
        sale  = sum(mm.x[i, j] for (i, j) in mm.A if i == n)
        entra = sum(mm.x[i, j] for (i, j) in mm.A if j == n)
        return sale - entra == q2[n]
    mm.bal = pyo.Constraint(mm.N, rule=bal)
    r = pyo.SolverFactory("appsi_highs").solve(mm)
    if str(r.solver.termination_condition) != "optimal":
        return None, None
    return pyo.value(mm.obj), {a: pyo.value(mm.x[a]) for a in mm.A}


# TODO: use resolver() para responder las preguntas de sensibilidad de su instancia.
# Ejemplo de uso: aumentar en una unidad la capacidad de un arco y medir el ahorro.


## Paso 9. Respuestas y limitaciones

Responda aquí, en prosa, las preguntas de su instancia. Cierre con la limitación del modelo que se
pide en la sección 4 del enunciado: un supuesto que el modelo hace, que la realidad no cumple, y en
qué dirección sesga la conclusión.

---

**Recordatorio de entrega:** repositorio con `README.md`, `datos/`, `modelo.py`,
`resultados/` y este cuaderno. Domingo 13 de septiembre, 23:59.
